# Week 9 and 10
## Telco Customer Churn — Complete Classification Workflow

### Concepts applied
- Train/test split
- Cross-validation
- Confusion matrix
- Precision, Recall and F1-score
- ROC curve and ROC-AUC
- Feature engineering
- Feature scaling
- Handling missing data
- Categorical encoding
- Logistic Regression and Random Forest comparison

**Dataset:** Telco Customer Churn  
**Target:** `Churn` (`Yes` = customer churned, `No` = customer stayed)

## 1. Import libraries
We use pandas/numpy for data handling, matplotlib for plots, and scikit-learn for preprocessing, modeling and evaluation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

pd.set_option('display.max_columns', None)

## 2. Load the dataset
In Google Colab, upload `Telco-Customer-Churn.csv` when prompted. The fallback path also works if the CSV is already present in `/content/`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    csv_name = next(iter(uploaded))
    df = pd.read_csv(csv_name)
except Exception:
    df = pd.read_csv('/content/Telco-Customer-Churn.csv')

print('Dataset shape:', df.shape)
df.head()

## 3. Initial data inspection
We inspect data types, duplicates, target distribution and apparent missing values.

In [ ]:
print(df.info())
print('
Duplicate rows:', df.duplicated().sum())
print('
Missing values before cleaning:')
print(df.isnull().sum())
print('
Churn distribution:')
print(df['Churn'].value_counts())
print('
Churn percentage:')
print((df['Churn'].value_counts(normalize=True) * 100).round(2))

## 4. Handle hidden missing values
`TotalCharges` is read as an object/string column because some rows contain blank strings. We convert it to numeric using `errors='coerce'`, which turns invalid/blank entries into `NaN`.

We will **not** fill these values manually before splitting the data. Instead, median imputation is placed inside the preprocessing pipeline so information from the test set cannot leak into training.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('Missing TotalCharges after numeric conversion:', df['TotalCharges'].isna().sum())
print('
All missing values after conversion:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 5. Feature engineering
We create meaningful new features from existing columns:

- **TotalServices:** number of major telecom services used by a customer.
- **AvgChargePerMonth:** average historical charge per month (`TotalCharges / tenure`). For tenure 0, we avoid division by zero.
- **TenureGroup:** converts tenure into useful customer-lifecycle categories.

We also drop `customerID` because it is only an identifier and should not help predict churn.

In [ ]:
service_columns = [
    'PhoneService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]

df['TotalServices'] = sum((df[col] == 'Yes').astype(int) for col in service_columns)

df['AvgChargePerMonth'] = np.where(
    df['tenure'] > 0,
    df['TotalCharges'] / df['tenure'],
    np.nan
)

df['TenureGroup'] = pd.cut(
    df['tenure'],
    bins=[-1, 12, 24, 48, 60, np.inf],
    labels=['0-12', '13-24', '25-48', '49-60', '61+']
)

df = df.drop(columns=['customerID'])

df[['tenure', 'TotalCharges', 'TotalServices', 'AvgChargePerMonth', 'TenureGroup']].head()

## 6. Separate features and target
The target is converted from `Yes/No` to `1/0`.

In [ ]:
X = df.drop(columns='Churn')
y = df['Churn'].map({'No': 0, 'Yes': 1})

print('X shape:', X.shape)
print('y shape:', y.shape)
print('
Target counts:')
print(y.value_counts())

## 7. Train/test split
We use an **80/20 stratified split**. Stratification keeps approximately the same churn/non-churn proportion in both training and test sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print('Training set:', X_train.shape)
print('Test set:', X_test.shape)
print('
Training churn rate:', round(y_train.mean(), 4))
print('Test churn rate:', round(y_test.mean(), 4))

## 8. Preprocessing: missing data, scaling and encoding
We automatically identify numerical and categorical columns.

### Numerical pipeline
1. Fill missing values using the **median**.
2. Apply **StandardScaler** so numerical features are on a comparable scale.

### Categorical pipeline
1. Fill missing values using the **most frequent** category.
2. Apply **OneHotEncoder** to convert categories into machine-readable binary columns.

Using a `Pipeline` ensures these transformations are fitted only on training folds during cross-validation, which prevents data leakage.

In [ ]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=['int64', 'float64']).columns.tolist()

print('Numerical features:', numeric_features)
print('
Categorical features:', categorical_features)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

## 9. Logistic Regression model
Logistic Regression is a strong baseline for binary classification and produces probabilities that can be used for ROC-AUC analysis.

In [ ]:
log_reg_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

log_reg_model.fit(X_train, y_train)
print('Logistic Regression training complete.')

## 10. Cross-validation
We use **5-fold Stratified Cross-Validation** on the training data. Each fold preserves the class proportion.

We evaluate Accuracy, Precision, Recall, F1 and ROC-AUC across the folds.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

cv_results = cross_validate(
    log_reg_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

cv_summary = pd.DataFrame({
    metric: [cv_results[f'test_{metric}'].mean(), cv_results[f'test_{metric}'].std()]
    for metric in scoring
}, index=['Mean', 'Std']).T

cv_summary.round(4)

## 11. Predictions on the unseen test set
Class predictions are used for the confusion matrix and classification metrics. Predicted probabilities are used for ROC-AUC and the ROC curve.

In [ ]:
y_pred = log_reg_model.predict(X_test)
y_prob = log_reg_model.predict_proba(X_test)[:, 1]

print('Predictions generated.')

## 12. Confusion matrix
The confusion matrix shows:
- **True Negative (TN):** correctly predicted non-churn customers
- **False Positive (FP):** predicted churn but customer stayed
- **False Negative (FN):** predicted stay but customer churned
- **True Positive (TP):** correctly predicted churn customers

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print('Confusion Matrix:
', cm)

ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Churn', 'Churn']).plot()
plt.title('Logistic Regression - Confusion Matrix')
plt.show()

## 13. Precision, Recall, F1-score and Accuracy
- **Precision:** Of all customers predicted to churn, how many actually churned?
- **Recall:** Of all customers who actually churned, how many did the model identify?
- **F1-score:** Harmonic mean of precision and recall.
- **Accuracy:** Overall proportion of correct predictions.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

metrics_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Score': [accuracy, precision, recall, f1]
})

print(metrics_df.round(4).to_string(index=False))
print('
Detailed Classification Report:
')
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 14. ROC curve and ROC-AUC
The ROC curve plots **True Positive Rate** against **False Positive Rate** at different classification thresholds.

**ROC-AUC** summarizes this curve into one number:
- 0.5 ≈ random classifier
- closer to 1.0 = better discrimination

In [ ]:
auc_score = roc_auc_score(y_test, y_prob)
fpr, tpr, thresholds = roc_curve(y_test, y_prob)

print(f'ROC-AUC Score: {auc_score:.4f}')

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f'Logistic Regression (AUC = {auc_score:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 15. Random Forest comparison
To make the evaluation more meaningful, we train a second classifier. Random Forest does not require scaling mathematically, but keeping the same preprocessing pipeline gives both models the same clean and encoded inputs.

In [ ]:
rf_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight='balanced'
    ))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print('Random Forest training complete.')

## 16. Compare both models on the test set
This table compares the two classifiers using all major evaluation metrics.

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred),
        'Recall': recall_score(y_true, y_pred),
        'F1-score': f1_score(y_true, y_pred),
        'ROC-AUC': roc_auc_score(y_true, y_prob)
    }

comparison = pd.DataFrame([
    evaluate_model('Logistic Regression', y_test, y_pred, y_prob),
    evaluate_model('Random Forest', y_test, rf_pred, rf_prob)
]).set_index('Model')

comparison.round(4)

## 17. ROC curve comparison

In [ ]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_prob)

plt.figure(figsize=(7, 5))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={roc_auc_score(y_test, y_prob):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC={roc_auc_score(y_test, rf_prob):.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve Comparison')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 18. Final conclusion
This notebook applied a complete supervised machine-learning classification workflow to the Telco Customer Churn dataset.

### Concepts demonstrated
1. **Missing-data handling:** hidden blank values in `TotalCharges` were converted to `NaN` and imputed using the training-data median inside a pipeline.
2. **Feature engineering:** `TotalServices`, `AvgChargePerMonth`, and `TenureGroup` were created.
3. **Feature scaling:** numerical features were standardized using `StandardScaler`.
4. **Categorical encoding:** categorical columns were converted using one-hot encoding.
5. **Train/test split:** an 80/20 stratified split was used.
6. **Cross-validation:** 5-fold stratified CV measured model stability.
7. **Confusion matrix:** correct and incorrect churn predictions were examined.
8. **Precision, Recall and F1:** classification quality was measured beyond accuracy.
9. **ROC-AUC:** probability-based discrimination was evaluated.
10. **Model comparison:** Logistic Regression was compared with Random Forest.

For a churn problem, **Recall and F1-score for the churn class are especially useful**, because missing customers who are likely to leave can be costly to a company. ROC-AUC additionally measures how well the model separates churners from non-churners across different decision thresholds.